First training runs - DeepLabV3+ : ImageNet only vs local vs DeepGlobe subset vs DeepGlobe subset + local
Colab: Runtime > Change runtime type > T4 GPU. Paste each "# %%" block into its own cell.
Run cells 1-3 FIRST (DeepGlobe) while QGIS is still exporting your tiles.


In [ ]:
# %% 1. Install + common code
!pip -q install segmentation-models-pytorch tifffile kaggle
import os, glob, copy, random, numpy as np, torch, tifffile, matplotlib.pyplot as plt
from PIL import Image
import segmentation_models_pytorch as smp
torch.manual_seed(0); np.random.seed(0); random.seed(0)
dev = "cuda"
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])  # ImageNet stats, RGB order

def to_tensor(img, m):
    x = ((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)
    return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(m.copy(), dtype=torch.float32)[None]

def augment(img, m):
    k = np.random.randint(4); img, m = np.rot90(img, k), np.rot90(m, k)
    if np.random.rand() < .5: img, m = img[:, ::-1], m[:, ::-1]
    return img, m

def new_model():
    return smp.DeepLabV3Plus("resnet34", encoder_weights="imagenet", classes=1).to(dev)

dice = smp.losses.DiceLoss("binary"); bce = torch.nn.BCEWithLogitsLoss()

def train(model, loader, epochs, lr=1e-4, eval_loader=None, tag=""):
    opt = torch.optim.AdamW(model.parameters(), lr); hist = []
    for ep in range(epochs):
        model.train(); tot = 0
        for x, y in loader:
            x, y = x.to(dev), y.to(dev); out = model(x)
            loss = dice(out, y) + bce(out, y)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        h = {"loss": tot / len(loader)}
        if eval_loader is not None: h.update(evaluate(model, eval_loader))
        hist.append(h)
        print(f"[{tag}] ep {ep+1:2d} " + " ".join(f"{k} {v:.3f}" for k, v in h.items()))
    return hist

def evaluate(model, loader, thr=0.5):
    model.eval(); tp = fp = fn = 0
    with torch.no_grad():
        for x, y in loader:
            p = (torch.sigmoid(model(x.to(dev))) > thr).float().cpu()
            tp += (p * y).sum().item(); fp += (p * (1 - y)).sum().item(); fn += ((1 - p) * y).sum().item()
    P = tp / (tp + fp + 1e-9); R = tp / (tp + fn + 1e-9)
    return dict(precision=P, recall=R, f1=2 * P * R / (P + R + 1e-9), iou=tp / (tp + fp + fn + 1e-9))

In [ ]:
# %% 2. Download DeepGlobe (needs kaggle.json: kaggle.com > Settings > API > Create New Token)
from google.colab import files
files.upload()                                  # pick kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d balraj98/deepglobe-road-extraction-dataset -p /content/dg -q
!cd /content/dg && unzip -q -o '*.zip' 'train/*' && ls train | head -4 && ls train | wc -l

In [ ]:
# %% 3. Train on a DeepGlobe subset (random 512 px crops keep the 0.5 m resolution)
sats = sorted(glob.glob("/content/dg/train/*_sat.jpg"))
random.shuffle(sats); sats = sats[:400]
dg_data = []
for s in sats:
    img = np.array(Image.open(s).convert("RGB"))
    m = (np.array(Image.open(s.replace("_sat.jpg", "_mask.png")).convert("L")) > 128).astype(np.float32)
    dg_data.append((img, m))

class DGCrops(torch.utils.data.Dataset):
    def __len__(self): return len(dg_data)
    def __getitem__(self, i):
        img, m = dg_data[i]; y0, x0 = np.random.randint(0, 1024 - 512, 2)
        return to_tensor(*augment(img[y0:y0+512, x0:x0+512], m[y0:y0+512, x0:x0+512]))

dg_loader = torch.utils.data.DataLoader(DGCrops(), batch_size=8, shuffle=True, num_workers=2)
dg_model = new_model()
dg_hist = train(dg_model, dg_loader, epochs=10, tag="DeepGlobe subset")
torch.save(dg_model.state_dict(), "dg_subset.pt")

In [ ]:
# %% 4. Upload your tiles (zip containing images/ and masks/)
files.upload()                                  # pick dataset.zip
!unzip -q -o "$(ls *.zip | grep -i dataset | head -1)" -d data
root = "data" if os.path.isdir("data/images") else glob.glob("data/*/images")[0].rsplit("/images", 1)[0]

In [ ]:
# %% 5. Local tiles + spatial split by grid column (easternmost ~20% = test area)
names = sorted(os.path.basename(p) for p in glob.glob(f"{root}/images/*.tif"))
names = [n for n in names if os.path.exists(f"{root}/masks/{n}")]
names = [n for n in names if (tifffile.imread(f"{root}/masks/{n}") > 0).mean() > 0.005]  # keep tiles with mapped roads
def load(n):
    img = tifffile.imread(f"{root}/images/{n}")
    if img.shape[0] == 3: img = img.transpose(1, 2, 0)
    return img[..., :3].astype(np.uint8), (tifffile.imread(f"{root}/masks/{n}") > 0).astype(np.float32)
gx = {n: int(n.rsplit(".", 1)[0].split("_")[-2]) for n in names}
cols = sorted(set(gx.values())); cut = cols[int(len(cols) * 0.8)]
train_n = [n for n in names if gx[n] < cut]; test_n = [n for n in names if gx[n] >= cut]
print(f"{len(names)} tiles | train {len(train_n)} | test {len(test_n)}")

class Tiles(torch.utils.data.Dataset):
    def __init__(self, ns, aug=False): self.d = [load(n) for n in ns]; self.aug = aug
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        img, m = self.d[i]
        return to_tensor(*(augment(img, m) if self.aug else (img, m)))
tr = torch.utils.data.DataLoader(Tiles(train_n, aug=True), batch_size=4, shuffle=True)
te = torch.utils.data.DataLoader(Tiles(test_n), batch_size=4)

In [ ]:
# %% 6. The four models, all evaluated on the same test area
results = {}
results["ImageNet only"] = evaluate(new_model(), te)
results["DeepGlobe subset only"] = evaluate(dg_model, te)

local_model = new_model()
local_hist = train(local_model, tr, epochs=25, eval_loader=te, tag="local only")
results["Local tiles only"] = local_hist[-1]

ft_model = copy.deepcopy(dg_model)
ft_hist = train(ft_model, tr, epochs=25, eval_loader=te, tag="DeepGlobe subset + local")
results["DeepGlobe subset + local"] = ft_hist[-1]

for k, v in results.items():
    print(f"{k:28s} " + " ".join(f"{m} {v[m]:.3f}" for m in ["precision", "recall", "f1", "iou"]))

In [ ]:
# %% 7. Figures for the slides
os.makedirs("figs", exist_ok=True)
ms = ["precision", "recall", "f1", "iou"]; xs = np.arange(len(ms)); w = 0.2
plt.figure(figsize=(9, 4.5))
for i, (k, v) in enumerate(results.items()):
    plt.bar(xs + (i - 1.5) * w, [v[m] for m in ms], w, label=k)
plt.xticks(xs, ["Precision", "Recall", "F1", "IoU"]); plt.ylim(0, 1); plt.legend(fontsize=9)
plt.title("Test area, OSM-derived labels"); plt.tight_layout(); plt.savefig("figs/comparison.png", dpi=200)

plt.figure(figsize=(10, 4))
e = range(1, 26)
plt.subplot(1, 2, 1)
plt.plot(e, [h["loss"] for h in local_hist], label="Local only"); plt.plot(e, [h["loss"] for h in ft_hist], label="DeepGlobe subset + local")
plt.title("Training loss"); plt.xlabel("Epoch"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(e, [h["f1"] for h in local_hist], label="Local only"); plt.plot(e, [h["f1"] for h in ft_hist], label="DeepGlobe subset + local")
plt.title("Test-area F1"); plt.xlabel("Epoch"); plt.legend(); plt.tight_layout(); plt.savefig("figs/curves.png", dpi=200)

# Prediction panels: image | OSM mask | DeepGlobe subset only | DeepGlobe subset + local
ds = te.dataset; idx = np.argsort([-d[1].mean() for d in ds.d])[:4]
fig, ax = plt.subplots(len(idx), 4, figsize=(12, 3 * len(idx)))
for r, i in enumerate(idx):
    x, y = ds[i]
    with torch.no_grad():
        dg_model.eval(); ft_model.eval()
        p1 = torch.sigmoid(dg_model(x[None].to(dev)))[0, 0].cpu().numpy()
        p2 = torch.sigmoid(ft_model(x[None].to(dev)))[0, 0].cpu().numpy()
    for a, im, t in zip(ax[r], [ds.d[i][0], y[0], p1 > .5, p2 > .5], ["Image", "OSM mask", "DeepGlobe subset", "DeepGlobe subset + local"]):
        a.imshow(im, cmap=None if t == "Image" else "gray"); a.set_title(t, fontsize=9); a.axis("off")
plt.tight_layout(); plt.savefig("figs/predictions.png", dpi=150)

import json; json.dump(results, open("figs/results.json", "w"), indent=2)
!zip -q -r figs.zip figs
files.download("figs.zip")